In [2]:
import sys
sys.path.append('/home/xilinx')

# Needed to run inference on TCU
import time
import numpy as np
import pynq
from pynq import Overlay
from tcu_pynq.driver import Driver
from tcu_pynq.architecture import pynqz1

# Needed for unpacking and displaying image data
%matplotlib inline
import matplotlib.pyplot as plt
import pickle

In [3]:
overlay = Overlay('/home/xilinx/tensil_pynqz1.bit')
tcu = Driver(pynqz1, overlay.axi_dma_0, dma_buffer_size=512*1024)

# The Tensil PYNQ driver includes the PYNQ Z1 architecture definition. Here it is in an excerpt from architecture.py: you can see that it matches the architecture we used previously.

In [41]:
# open data in Pynq
data = pickle.load(open('/home/xilinx/test_data_verified_32x32.pickle', 'rb'))
labels = pickle.load(open('/home/xilinx/test_labels_verified_32x32.pickle', 'rb'))

print(f"shape: {data.shape}")  # (N, 1, 32, 32)

label_names = ['monocyte', 'granulocyte', 'lymphocyte']


# test data, no further process
for i in range(12):
    img = data[i]  # [1, 32, 32]
    img = np.transpose(img, (1, 2, 0))  # [32, 32, 1]
    img = np.pad(img, [(0, 0), (0, 0), (0, 7)], 'constant', constant_values=0)  # [32, 32, 8]
    img_tensil = img.reshape((-1, 8))  # [1024, 8]
    
    outputs = tcu.run({'input': img_tensil})
    pred = np.argmax(outputs['output'][:4])
    
    print(f"[{i}] {label_names[labels[i]]:12s} -> {label_names[pred]:12s}")

shape: (400, 1, 32, 32)
[0] lymphocyte   -> lymphocyte  
[1] lymphocyte   -> lymphocyte  
[2] lymphocyte   -> lymphocyte  
[3] lymphocyte   -> monocyte    
[4] lymphocyte   -> lymphocyte  
[5] lymphocyte   -> lymphocyte  
[6] lymphocyte   -> lymphocyte  
[7] lymphocyte   -> lymphocyte  
[8] lymphocyte   -> lymphocyte  
[9] lymphocyte   -> lymphocyte  
[10] lymphocyte   -> lymphocyte  
[11] lymphocyte   -> lymphocyte  


In [42]:
# ============================================================================
# Pynq board full test script - 3-class classification m-g-(b-t)
# ============================================================================
import numpy as np
import time
import pickle

print("="*80)
print("Tensil Inference Test - Uniform Sampling (3 classes)")
print("="*80)

# ============================================================================
# 0. Load model (32ch)
# ============================================================================

MODEL_NAME = 'resnet10_new_tensil_fp32_16ch_32x32_onnx_pynqz1'

tcu.load_model(f'/home/xilinx/{MODEL_NAME}.tmodel')

# ============================================================================
# 1. Load Data
# ============================================================================
print("\n1. Loading test data...")

with open('/home/xilinx/test_data_verified_32x32.pickle', 'rb') as f:
    data = pickle.load(f)

with open('/home/xilinx/test_labels_verified_32x32.pickle', 'rb') as f:
    labels = pickle.load(f)

print(f"  Data shape: {data.shape}")
print(f"  Label shape: {labels.shape}")
print(f"  Data range: [{data.min():.4f}, {data.max():.4f}]")

# ============================================================================
# 2. Uniform Distribution Sampling
# ============================================================================
print("\n" + "="*80)
print("2. Uniform Distribution Sampling (3 classes)")
print("="*80)

label_names = ['monocyte', 'granulocyte', 'lymphocyte']
NUM_CLASSES = 3

# Check data distribution
print(f"\nFull dataset label distribution:")
for i, name in enumerate(label_names):
    count = np.sum(labels == i)
    indices = np.where(labels == i)[0]
    
    print(f"  {name:25s} (idx={i}): {count:3d} images", end='')
    if count > 0:
        print(f" | Sample indices: {indices[:5].tolist()}")
    else:
        print(f" | ⚠️ No samples")

# Uniform sampling: select N images per class
samples_per_class = 100  # Select 50 images per class
selected_indices = []
selected_labels = []

print(f"\nUniform sampling ({samples_per_class} images per class):")

for class_idx in range(NUM_CLASSES):
    class_indices = np.where(labels == class_idx)[0]
    
    if len(class_indices) >= samples_per_class:
        # Evenly spaced sampling (more representative)
        step = len(class_indices) // samples_per_class
        selected = class_indices[::step][:samples_per_class]
    else:
        # Select all
        selected = class_indices
    
    selected_indices.extend(selected.tolist())
    selected_labels.extend([class_idx] * len(selected))
    
    print(f"  {label_names[class_idx]:25s}: Selected {len(selected)} images")

selected_indices = np.array(selected_indices)
selected_labels = np.array(selected_labels)

print(f"\nTotal: {len(selected_indices)} images")
print(f"Label distribution: {np.bincount(selected_labels)}")

# Extract data
data_test = data[selected_indices]
labels_test = selected_labels

print(f"✓ Sampling complete")

# ============================================================================
# 3. Data Format Conversion (Tensil format)
# ============================================================================
print("\n" + "="*80)
print("3. Data format conversion")
print("="*80)

def get_img(data, n):
    """
    Convert to Tensil TCU format
    Input: [1, 32, 32] (NCHW)
    Output: [1024, 8] (H*W, array_size)
    """
    img = data[n]  # [1, 32, 32]
    img = np.transpose(img, (1, 2, 0))  # [32, 32, 1]
    # Pad to array_size=8 (pynqz1 architecture)
    img = np.pad(img, [(0, 0), (0, 0), (0, 7)], 'constant', constant_values=0)
    return img.reshape((-1, 8))  # [1024, 8]

# Test conversion
sample_img = get_img(data_test, 0)
print(f"  Input shape: {data_test[0].shape} → Tensil format: {sample_img.shape}")
print(f"✓ Format conversion function ready")

# ============================================================================
# 4. Inference Test
# ============================================================================
print("\n" + "="*80)
print("4. Tensil Inference (Uniform sampling, 3 classes)")
print("="*80)

correct = 0
times = []
all_logits = []

for i in range(len(data_test)):
    img = get_img(data_test, i)
    true_idx = labels_test[i]
    
    start = time.time()
    outputs = tcu.run({'input': img})
    times.append(time.time() - start)
    
    logits = outputs['output'][:NUM_CLASSES]
    all_logits.append(logits)
    pred_idx = np.argmax(logits)
    
    correct += (pred_idx == true_idx)
    status = "✓" if pred_idx == true_idx else "✗"
    
    # Show first 20 and last 5
    if i < 20 or i >= len(data_test) - 5:
        print(f"  [{i:3d}] {status} True:{label_names[true_idx]:25s}(idx={true_idx}) → Pred:{label_names[pred_idx]:25s}(idx={pred_idx}) | {times[-1]*1000:.1f}ms")
        print(f"        Logits: [{logits[0]:7.4f}, {logits[1]:7.4f}, {logits[2]:7.4f}]")
    elif i == 20:
        print(f"  ... (omitted {len(data_test)-25} samples) ...")

all_logits = np.array(all_logits)

times = np.array(times)
std_time = np.std(times) * 1000

# ============================================================================
# 5. Statistical Analysis
# ============================================================================
print("\n" + "="*80)
print("5. Test Results Statistics")
print("="*80)

accuracy = correct / len(data_test) * 100
avg_time = np.mean(times) * 1000

print(f"  Total samples: {len(data_test)}")
print(f"  Accuracy:   {correct}/{len(data_test)} = {accuracy:.1f}%")
print(f"  Average time: {avg_time:.2f} ms")
print(f"  Throughput:   {1000/avg_time:.1f} FPS")

# Per-class accuracy
print(f"\nPer-class Accuracy:")
for class_idx in range(NUM_CLASSES):
    class_mask = (labels_test == class_idx)
    if np.any(class_mask):
        class_preds = np.argmax(all_logits[class_mask], axis=1)
        class_correct = np.sum(class_preds == class_idx)
        class_total = np.sum(class_mask)
        class_acc = class_correct / class_total * 100
        
        print(f"  {label_names[class_idx]:25s}: {class_correct:3d}/{class_total:3d} = {class_acc:5.1f}%")

# Confusion Matrix
print(f"\nConfusion Matrix:")
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for i in range(len(data_test)):
    true_idx = labels_test[i]
    pred_idx = np.argmax(all_logits[i])
    confusion[true_idx][pred_idx] += 1

# Header
header = "True/Pred".ljust(27)
for name in label_names:
    header += f"{name[:15]:>15s}"
print(f"  {header}")
print(f"  " + "-"*72)

# Data rows
for i, true_name in enumerate(label_names):
    row = f"  {true_name:<25s}"
    for j in range(NUM_CLASSES):
        row += f"{confusion[i][j]:>15d}"
    print(row)

# Logits analysis
print(f"\nAverage per-class Logits:")
for class_idx in range(NUM_CLASSES):
    class_mask = (labels_test == class_idx)
    if np.any(class_mask):
        avg_logits = np.mean(all_logits[class_mask], axis=0)
        print(f"  {label_names[class_idx]:25s}: [{avg_logits[0]:7.4f}, {avg_logits[1]:7.4f}, {avg_logits[2]:7.4f}]")
        print(f"                             Strongest: idx={np.argmax(avg_logits)} ({label_names[np.argmax(avg_logits)]})")

# ============================================================================
# 6. Summary
# ============================================================================
print("\n" + "="*80)
print("6. Deployment Summary")
print("="*80)

print(f"✅ Tensil deployment very successful!")
print(f"   Accuracy: {accuracy:.1f}%")
print(f"   Latency std dev: {std_time:.2f} ms")
print(f"   Performance: {1000/avg_time:.1f} FPS")

Tensil Inference Test - Uniform Sampling (3 classes)

1. Loading test data...
  Data shape: (400, 1, 32, 32)
  Label shape: (400,)
  Data range: [-1.4531, 3.3464]

2. Uniform Distribution Sampling (3 classes)

Full dataset label distribution:
  monocyte                  (idx=0): 100 images | Sample indices: [200, 201, 202, 203, 204]
  granulocyte               (idx=1): 100 images | Sample indices: [300, 301, 302, 303, 304]
  lymphocyte                (idx=2): 200 images | Sample indices: [0, 1, 2, 3, 4]

Uniform sampling (100 images per class):
  monocyte                 : Selected 100 images
  granulocyte              : Selected 100 images
  lymphocyte               : Selected 100 images

Total: 300 images
Label distribution: [100 100 100]
✓ Sampling complete

3. Data format conversion
  Input shape: (1, 32, 32) → Tensil format: (1024, 8)
✓ Format conversion function ready

4. Tensil Inference (Uniform sampling, 3 classes)
  [  0] ✓ True:monocyte                 (idx=0) → Pred:monocyt

In [43]:
# ============================================================================
# Pynq board full test script - 3-class classification m-g-(b-t)
# ============================================================================
import numpy as np
import time
import pickle

print("="*80)
print("Tensil Inference Test - Uniform Sampling (3 classes)")
print("="*80)

# ============================================================================
# 0. Load model (16ch)
# ============================================================================

MODEL_NAME = 'resnet10_tensil_fp32_16ch_32x32_onnx_pynqz1'

tcu.load_model(f'/home/xilinx/{MODEL_NAME}.tmodel')

# ============================================================================
# 1. Load Data
# ============================================================================
print("\n1. Loading test data...")

with open('/home/xilinx/test_data_verified_32x32.pickle', 'rb') as f:
    data = pickle.load(f)

with open('/home/xilinx/test_labels_verified_32x32.pickle', 'rb') as f:
    labels = pickle.load(f)

print(f"  Data shape: {data.shape}")
print(f"  Label shape: {labels.shape}")
print(f"  Data range: [{data.min():.4f}, {data.max():.4f}]")

# ============================================================================
# 2. Uniform Distribution Sampling
# ============================================================================
print("\n" + "="*80)
print("2. Uniform Distribution Sampling (3 classes)")
print("="*80)

label_names = ['monocyte', 'granulocyte', 'lymphocyte']
NUM_CLASSES = 3

# Check data distribution
print(f"\nFull dataset label distribution:")
for i, name in enumerate(label_names):
    count = np.sum(labels == i)
    indices = np.where(labels == i)[0]
    
    print(f"  {name:25s} (idx={i}): {count:3d} images", end='')
    if count > 0:
        print(f" | Sample indices: {indices[:5].tolist()}")
    else:
        print(f" | ⚠️ No samples")

# Uniform sampling: select N images per class
samples_per_class = 100  # Select 50 images per class
selected_indices = []
selected_labels = []

print(f"\nUniform sampling ({samples_per_class} images per class):")

for class_idx in range(NUM_CLASSES):
    class_indices = np.where(labels == class_idx)[0]
    
    if len(class_indices) >= samples_per_class:
        # Evenly spaced sampling (more representative)
        step = len(class_indices) // samples_per_class
        selected = class_indices[::step][:samples_per_class]
    else:
        # Select all
        selected = class_indices
    
    selected_indices.extend(selected.tolist())
    selected_labels.extend([class_idx] * len(selected))
    
    print(f"  {label_names[class_idx]:25s}: Selected {len(selected)} images")

selected_indices = np.array(selected_indices)
selected_labels = np.array(selected_labels)

print(f"\nTotal: {len(selected_indices)} images")
print(f"Label distribution: {np.bincount(selected_labels)}")

# Extract data
data_test = data[selected_indices]
labels_test = selected_labels

print(f"✓ Sampling complete")

# ============================================================================
# 3. Data Format Conversion (Tensil format)
# ============================================================================
print("\n" + "="*80)
print("3. Data format conversion")
print("="*80)

def get_img(data, n):
    """
    Convert to Tensil TCU format
    Input: [1, 32, 32] (NCHW)
    Output: [1024, 8] (H*W, array_size)
    """
    img = data[n]  # [1, 32, 32]
    img = np.transpose(img, (1, 2, 0))  # [32, 32, 1]
    # Pad to array_size=8 (pynqz1 architecture)
    img = np.pad(img, [(0, 0), (0, 0), (0, 7)], 'constant', constant_values=0)
    return img.reshape((-1, 8))  # [1024, 8]

# Test conversion
sample_img = get_img(data_test, 0)
print(f"  Input shape: {data_test[0].shape} → Tensil format: {sample_img.shape}")
print(f"✓ Format conversion function ready")

# ============================================================================
# 4. Inference Test
# ============================================================================
print("\n" + "="*80)
print("4. Tensil Inference (Uniform sampling, 3 classes)")
print("="*80)

correct = 0
times = []
all_logits = []

for i in range(len(data_test)):
    img = get_img(data_test, i)
    true_idx = labels_test[i]
    
    start = time.time()
    outputs = tcu.run({'input': img})
    times.append(time.time() - start)
    
    logits = outputs['output'][:NUM_CLASSES]
    all_logits.append(logits)
    pred_idx = np.argmax(logits)
    
    correct += (pred_idx == true_idx)
    status = "✓" if pred_idx == true_idx else "✗"
    
    # Show first 20 and last 5
    if i < 20 or i >= len(data_test) - 5:
        print(f"  [{i:3d}] {status} True:{label_names[true_idx]:25s}(idx={true_idx}) → Pred:{label_names[pred_idx]:25s}(idx={pred_idx}) | {times[-1]*1000:.1f}ms")
        print(f"        Logits: [{logits[0]:7.4f}, {logits[1]:7.4f}, {logits[2]:7.4f}]")
    elif i == 20:
        print(f"  ... (omitted {len(data_test)-25} samples) ...")

all_logits = np.array(all_logits)

times = np.array(times)
std_time = np.std(times) * 1000

# ============================================================================
# 5. Statistical Analysis
# ============================================================================
print("\n" + "="*80)
print("5. Test Results Statistics")
print("="*80)

accuracy = correct / len(data_test) * 100
avg_time = np.mean(times) * 1000

print(f"  Total samples: {len(data_test)}")
print(f"  Accuracy:   {correct}/{len(data_test)} = {accuracy:.1f}%")
print(f"  Average time: {avg_time:.2f} ms")
print(f"  Throughput:   {1000/avg_time:.1f} FPS")

# Per-class accuracy
print(f"\nPer-class Accuracy:")
for class_idx in range(NUM_CLASSES):
    class_mask = (labels_test == class_idx)
    if np.any(class_mask):
        class_preds = np.argmax(all_logits[class_mask], axis=1)
        class_correct = np.sum(class_preds == class_idx)
        class_total = np.sum(class_mask)
        class_acc = class_correct / class_total * 100
        
        print(f"  {label_names[class_idx]:25s}: {class_correct:3d}/{class_total:3d} = {class_acc:5.1f}%")

# Confusion Matrix
print(f"\nConfusion Matrix:")
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for i in range(len(data_test)):
    true_idx = labels_test[i]
    pred_idx = np.argmax(all_logits[i])
    confusion[true_idx][pred_idx] += 1

# Header
header = "True/Pred".ljust(27)
for name in label_names:
    header += f"{name[:15]:>15s}"
print(f"  {header}")
print(f"  " + "-"*72)

# Data rows
for i, true_name in enumerate(label_names):
    row = f"  {true_name:<25s}"
    for j in range(NUM_CLASSES):
        row += f"{confusion[i][j]:>15d}"
    print(row)

# Logits analysis
print(f"\nAverage per-class Logits:")
for class_idx in range(NUM_CLASSES):
    class_mask = (labels_test == class_idx)
    if np.any(class_mask):
        avg_logits = np.mean(all_logits[class_mask], axis=0)
        print(f"  {label_names[class_idx]:25s}: [{avg_logits[0]:7.4f}, {avg_logits[1]:7.4f}, {avg_logits[2]:7.4f}]")
        print(f"                             Strongest: idx={np.argmax(avg_logits)} ({label_names[np.argmax(avg_logits)]})")

# ============================================================================
# 6. Summary
# ============================================================================
print("\n" + "="*80)
print("6. Deployment Summary")
print("="*80)

print(f"✅ Tensil deployment very successful!")
print(f"   Accuracy: {accuracy:.1f}%")
print(f"   Latency std dev: {std_time:.2f} ms")
print(f"   Performance: {1000/avg_time:.1f} FPS")

Tensil Inference Test - Uniform Sampling (3 classes)

1. Loading test data...
  Data shape: (400, 1, 32, 32)
  Label shape: (400,)
  Data range: [-1.4531, 3.3464]

2. Uniform Distribution Sampling (3 classes)

Full dataset label distribution:
  monocyte                  (idx=0): 100 images | Sample indices: [200, 201, 202, 203, 204]
  granulocyte               (idx=1): 100 images | Sample indices: [300, 301, 302, 303, 304]
  lymphocyte                (idx=2): 200 images | Sample indices: [0, 1, 2, 3, 4]

Uniform sampling (100 images per class):
  monocyte                 : Selected 100 images
  granulocyte              : Selected 100 images
  lymphocyte               : Selected 100 images

Total: 300 images
Label distribution: [100 100 100]
✓ Sampling complete

3. Data format conversion
  Input shape: (1, 32, 32) → Tensil format: (1024, 8)
✓ Format conversion function ready

4. Tensil Inference (Uniform sampling, 3 classes)
  [  0] ✓ True:monocyte                 (idx=0) → Pred:monocyt